#Example 8.4.2 Implementing Post-Training Quantization with GPTQ

In [ ]:
#1: Load the Pre-Trained Model
# Install the necessary packages
!pip install torch torchvision

# Import necessary modules
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torchvision.models as models

# Load the pre-trained model (ResNet18 as an example)
model = models.resnet18(pretrained=True)
model.eval()  # Set model to evaluation mode

# Define the CIFAR-10 dataset and data transformations
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
])

test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)


In [ ]:
#2: Apply GPTQ for Post-Training Quantization
# Example of applying PyTorch's native quantization (simulating GPTQ)
import torch.quantization

# Prepare the model for static quantization
quantized_model = torch.quantization.quantize_dynamic(
    model, {torch.nn.Linear}, dtype=torch.qint8
)

# You could replace the above with a custom GPTQ implementation if available
# For example, if `apply_gptq` was defined:
# quantized_model = apply_gptq(model, bit_width=8)


In [ ]:
#3: Evaluate the Quantized Model
# Evaluate the quantized model on the CIFAR-10 test set
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = quantized_model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'Accuracy of the quantized model on the test dataset: {accuracy:.2f}%')


In [ ]:
#4: Deploy the Quantized Model to Google Cloud Platform (GCP)
#4.1 Save the quantized model
# Load the model from GCS for deployment
blob = bucket.blob('quantized_model.pth')
blob.download_to_filename('quantized_model.pth')

# Load the model into memory
model = models.resnet18()  # Initialize the model architecture
model.load_state_dict(torch.load('quantized_model.pth'))
model.eval()  # Set to evaluation mode
print("Quantized model loaded successfully.")

In [ ]:
#4.2 Load the Quantized Model for Deployment:
# Load the model from GCS for deployment
blob = bucket.blob('quantized_model.pth')
blob.download_to_filename('quantized_model.pth')

# Load the model into memory
model = models.resnet18()  # Initialize the model architecture
model.load_state_dict(torch.load('quantized_model.pth'))
model.eval()  # Set to evaluation mode
print("Quantized model loaded successfully.")

In [ ]:
#Optional: Deploy Using Vertex AI (GCP)
# Install the Google Cloud AI Platform SDK
!pip install google-cloud-aiplatform

# Initialize AI Platform
from google.cloud import aiplatform

# Set your project ID and region
project_id = 'your-project-id'  # Replace with your GCP project ID
location = 'us-central1'

# Initialize the AI platform
aiplatform.init(project=project_id, location=location)

# Upload the model to Vertex AI for serving
model = aiplatform.Model.upload(
    display_name="quantized-resnet18",
    artifact_uri="gs://your-bucket-name/quantized_model.pth",  # Your GCS model path
    serving_container_image_uri="us-docker.pkg.dev/vertex-ai/prediction/pytorch-cpu.1-8:latest"
)

# Deploy the model
endpoint = model.deploy(
    machine_type="n1-standard-4"
)

print(f"Model deployed at endpoint: {endpoint.display_name}")

#8.5 Distillation Methods for Model Compression

In [ ]:
#1: Install Required Libraries
!pip install torch torchvision

In [ ]:
#2: Train the Teacher Model
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torchvision.models as models

# Load and train the teacher model (ResNet18)
teacher_model = models.resnet18(pretrained=True)
teacher_model.eval()

# Define the dataset and data loader
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
])

# Download CIFAR-10 dataset
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)


In [ ]:
#3: Generate Soft Targets Using Temperature Scaling
# Generate soft targets using temperature scaling
def get_soft_targets(model, data_loader, temperature=3.0):
    soft_targets = []
    model.eval()
    with torch.no_grad():
        for images, _ in data_loader:
            outputs = model(images)
            # Apply temperature scaling
            soft_outputs = nn.Softmax(dim=1)(outputs / temperature)
            soft_targets.append(soft_outputs)
    return torch.cat(soft_targets)

# Generate soft targets from the teacher model
soft_targets = get_soft_targets(teacher_model, train_loader)

In [ ]:
#4: Train the Student Model Using Soft Targets
# Define a smaller student model
class StudentCNN(nn.Module):
    def __init__(self):
        super(StudentCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3)
        self.fc1 = nn.Linear(32 * 6 * 6, 120)
        self.fc2 = nn.Linear(120, 10)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.max_pool2d(x, 2)
        x = torch.relu(self.conv2(x))
        x = torch.max_pool2d(x, 2)
        x = x.view(-1, 32 * 6 * 6)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Initialize the student model
student_model = StudentCNN()

# Define the optimizer and loss function
optimizer = torch.optim.Adam(student_model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# Training loop
for epoch in range(10):
    student_model.train()
    for images, _ in train_loader:
        optimizer.zero_grad()
        outputs = student_model(images)
        loss = criterion(outputs, soft_targets[:outputs.shape[0]])  # Ensure batch size matches
        loss.backward()
        optimizer.step()
    print(f'Epoch {epoch+1}, Loss: {loss.item()}')


In [ ]:
#5: Evaluate the Student Model
# Load the test dataset
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

# Evaluate the student model
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        outputs = student_model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'Accuracy of the student model on the test dataset: {accuracy:.2f}%')

In [ ]:
#6: Save and Deploy the Model on GCP
# Authenticate to GCP
from google.colab import auth
auth.authenticate_user()

# Install the GCS Python library
!pip install google-cloud-storage
import torch
from google.cloud import storage

# Save the student model
torch.save(student_model.state_dict(), 'student_model.pth')

# Upload the saved model to Google Cloud Storage (GCS)
client = storage.Client()
bucket = client.bucket('your-bucket-name')  # Replace with your GCS bucket name
blob = bucket.blob('student_model.pth')
blob.upload_from_filename('student_model.pth')

print("Student model uploaded to GCS.")

